### Data Ingestion 

In [1]:
### Document Structure 

from pathlib import Path
from bs4 import BeautifulSoup
import time


In [2]:
# --------------------------------------------------
# 1. Load HTML
# --------------------------------------------------

start = time.time()

html_path = Path("../data/2d-game-creation-wokflow.html")

with html_path.open("r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "lxml")

print(f"HTML parsing: {time.time() - start:.4f}s")

HTML parsing: 0.0763s


In [3]:
# --------------------------------------------------
# 2. Find the actual article
# --------------------------------------------------

start = time.time()

h1 = soup.find("h1")

if h1 is None:
    raise ValueError("Could not find an <h1> in the HTML.")

article = h1.find_parent("div", class_="section")

if article is None:
    raise ValueError("Could not find the article <div class='section'>.")

print(f"Article identification: {time.time() - start:.4f}s")


Article identification: 0.0010s


In [4]:
# --------------------------------------------------
# 3. Extract structured blocks
# --------------------------------------------------

start = time.time()

blocks = []
current_section = None

elements = article.find_all([
    "h1",
    "h2",
    "h3",
    "h4",
    "p",
    "li",
    "pre",
    "table"
])

for element in elements:

    # ----------------------------------------------
    # Headings
    # ----------------------------------------------

    if element.name in ["h1", "h2", "h3", "h4"]:

        text = element.get_text(" ", strip=True)

        if not text:
            continue

        current_section = text

        blocks.append({
            "type": "heading",
            "tag": element.name,
            "section": current_section,
            "text": text
        })

    # ----------------------------------------------
    # Paragraphs
    # ----------------------------------------------

    elif element.name == "p":

        text = element.get_text(" ", strip=True)

        if not text:
            continue

        blocks.append({
            "type": "paragraph",
            "tag": "p",
            "section": current_section,
            "text": text
        })

    # ----------------------------------------------
    # Lists
    # ----------------------------------------------

    elif element.name == "li":

        text = element.get_text(" ", strip=True)

        if not text:
            continue

        blocks.append({
            "type": "list_item",
            "tag": "li",
            "section": current_section,
            "text": text
        })

    # ----------------------------------------------
    # Code blocks
    # ----------------------------------------------

    elif element.name == "pre":

        text = element.get_text("\n", strip=True)

        if not text:
            continue

        blocks.append({
            "type": "code",
            "tag": "pre",
            "section": current_section,
            "text": text
        })

    # ----------------------------------------------
    # Tables
    # ----------------------------------------------

    elif element.name == "table":

        rows = []

        for tr in element.find_all("tr"):

            cells = tr.find_all(["th", "td"])

            row = [
                cell.get_text(" ", strip=True)
                for cell in cells
            ]

            if row:
                rows.append(row)

        if not rows:
            continue

        # First row = table headers
        headers = rows[0]

        # Convert each table row into a semantic block
        for row in rows[1:]:

            if len(row) != len(headers):
                continue

            table_text = "\n".join(
                f"{header}: {value}"
                for header, value in zip(headers, row)
            )

            blocks.append({
                "type": "table_row",
                "tag": "table",
                "section": current_section,
                "text": table_text
            })


print(f"Block extraction: {time.time() - start:.4f}s")


# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print(f"\nExtracted {len(blocks)} blocks\n")

for i, block in enumerate(blocks):

    print(f"BLOCK {i}")
    print(f"TYPE: {block['type']}")
    print(f"SECTION: {block['section']}")
    print(f"TEXT:\n{block['text']}")
    print("-" * 80)

Block extraction: 0.0046s

Extracted 141 blocks

BLOCK 0
TYPE: list_item
SECTION: None
TEXT:
2D game development
--------------------------------------------------------------------------------
BLOCK 1
TYPE: list_item
SECTION: None
TEXT:
Get started with 2D game development
--------------------------------------------------------------------------------
BLOCK 2
TYPE: list_item
SECTION: None
TEXT:
2D game creation workflow
--------------------------------------------------------------------------------
BLOCK 3
TYPE: heading
SECTION: 2D game creation workflow
TEXT:
2D game creation workflow
--------------------------------------------------------------------------------
BLOCK 4
TYPE: paragraph
SECTION: 2D game creation workflow
TEXT:
Before you create a 2D game, you need to decide on a game perspective and an art style .
--------------------------------------------------------------------------------
BLOCK 5
TYPE: paragraph
SECTION: 2D game creation workflow
TEXT:
To create a 2D game, se